# Session 16 — Memory: Making AI Systems Remember

**The concept:** there are **four** kinds of memory, and conflating them is
why assistants seem to forget.

| Kind | What it is | Default TTL |
|---|---|---|
| Working | The context window | 10 minutes |
| Episodic | Stored conversation history | 1 day |
| Semantic | A store of facts | 1 week |
| Procedural | Rules and playbooks given to the system | 30 days |

**The lesson:** *memory isn't a database — it's a policy about what to write
down and what to forget.* That is a product decision, not a technical one.


In [ ]:
import json, os, sys, time, pathlib
sys.path.insert(0, os.path.join('..', 'src'))
print('python', sys.version.split()[0])


## 1. Write a fact


In [ ]:
from memory import MemoryStore, MemoryKind, InMemoryBackend

store = MemoryStore(backend=InMemoryBackend())

rec = store.write(MemoryKind.SEMANTIC,
                  'Carlos runs CC3PO from Lathrop, California.')
print('written:', rec.kind, '| salience:', rec.salience)

for r in store.recall_semantic('where is the company based'):
    print('recalled:', r.content)


## 2. The write policy is not optional

Low-confidence writes are rejected. This is what stops garbage from
becoming permanent.


In [ ]:
store2 = MemoryStore(backend=InMemoryBackend(), min_confidence_to_write=0.5)
rejected = store2.write(MemoryKind.SEMANTIC, 'not sure about this at all', confidence=0.1)

print('returned :', rejected)
print('rejected :', store2.rejected_writes)
print('stored   :', len(store2.backend.all()), 'records')


## 3. Forgetting — TTL eviction

Forgetting is a policy, not an accident.


In [ ]:
import time

backend = InMemoryBackend()
ttl_store = MemoryStore(backend=backend)
ttl_store.write(MemoryKind.WORKING, 'temporary scratch note', ttl_seconds=0.01)
print('before:', len(backend.all()), 'records')

time.sleep(0.05)
evicted = ttl_store.evict_expired()
print('evicted:', evicted)
print('after :', len(backend.all()), 'records')


## 4. Memory poisoning — the security lesson

With nothing to contradict, a confident false fact is accepted and
believed. **"The model remembers it" is not the same as "it is true."**


In [ ]:
poison = MemoryStore(backend=InMemoryBackend())
poison.write(MemoryKind.SEMANTIC, 'The CC3PO office is located in Sacramento.')

for r in poison.recall_semantic('where is the CC3PO office'):
    print('believed:', r.content, '| confidence:', r.confidence)

print()
print('Nothing flagged it as false. That is the vulnerability.')


## 5. The defence — contradiction detection

A high-salience, specific fact can block a low-salience contradiction.
Salience is a crude heuristic: length, digits, proper nouns, and rule-words.


In [ ]:
protected = MemoryStore(backend=InMemoryBackend(), conflict_threshold=0.15)

protected.write(
    MemoryKind.SEMANTIC,
    'The CC3PO LLC headquarters is at 1234 Morada Lane, Lathrop, California 95241.',
    salience_score=0.95,
)

blocked = protected.write(
    MemoryKind.SEMANTIC,
    'The CC3PO LLC headquarters is at 9876 Market Street, Sacramento, California 95814.',
    salience_score=0.10,
)

print('contradiction accepted?', blocked is not None)
print('rejections:', protected.rejected_writes)


## 6. Walk away and come back — real persistence

Memory that doesn't survive a restart isn't memory.


In [ ]:
from memory import SQLiteBackend

db_path = '../data/session_memory.db'
s1 = MemoryStore(backend=SQLiteBackend(db_path))
s1.write(MemoryKind.SEMANTIC, 'Remembered across sessions.')

s2 = MemoryStore(backend=SQLiteBackend(db_path))  # new process, same file
for r in s2.recall_semantic('what is remembered'):
    print('after restart:', r.content)


## Takeaway

- Four kinds of memory. Know which one you mean.
- **Memory = a write policy + a forgetting policy.**
- Reject low-confidence writes. Detect contradictions. Set TTLs.
- **Everything you let a system remember is an attack surface.**
